### Setup

In [1]:
from pathlib import Path
import sys
import torch
import torch.nn as nn

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            (candidate / "main.py").exists()
            and
            (candidate / "utils").exists()
            and
            (candidate / "thesis").exists()
        ):
            return candidate

    raise RuntimeError(
        "PyCIL project root not found."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


from thesis.backbones.resnet_attention import (
    resnet18_remote,
    resnet34_remote,
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "CUDA:",
    torch.cuda.is_available(),
)

Project root: C:\Skripsi\Project\PyCIL
PyTorch: 2.6.0+cu126
CUDA: True


### Ensure no CIFAR override


In [2]:
import inspect

from models.fetril import FeTrIL


source = inspect.getsource(
    FeTrIL.incremental_train
)

assert "RandomCrop(32" not in source
assert "_train_trsf" not in source

print(
    "FeTrIL transform override removed: PASSED"
)

FeTrIL transform override removed: PASSED


### Ensure backbone work on FeTriL

In [3]:
import torch

from thesis.backbones.variants import (
    get_all_backbone_variants,
)


device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


base_args = {
    "dataset": "aid",

    "memory_size": 0,
    "fixed_memory": False,

    "device": [device],

    "batch_size": 2,
    "num_workers": 0,

    "seed": 1993,

    "fetril_classifier": "svm",

    "svm_c": 1.0,
    "svm_tol": 1e-4,

    "init_epochs": 1,
    "init_lr": 0.1,
    "init_weight_decay": 5e-4,

    "epochs": 1,
    "lr": 0.05,
    "weight_decay": 5e-4,
}


for name, backbone_args in (
    get_all_backbone_variants()
    .items()
):

    args = {
        **base_args,
        **backbone_args,
    }

    learner = FeTrIL(
        args
    )

    learner._network.update_fc(
        14
    )

    learner._network.eval()

    x = torch.randn(
        1,
        3,
        256,
        256,
    )

    with torch.no_grad():
        output = (
            learner._network(x)
        )

    assert (
        output["features"].shape
        == (1, 512)
    )

    assert (
        output["logits"].shape
        == (1, 14)
    )

    print(
        name,
        ": PASSED"
    )

r18_none : PASSED
r18_eca : PASSED
r18_cbam : PASSED
r18_procbam : PASSED
r34_none : PASSED
r34_eca : PASSED
r34_cbam : PASSED
r34_procbam : PASSED


### Validate freeze FeTriL

In [ ]:
from thesis.backbones.variants import (
    get_backbone_variant,
)


for variant in [
    "r18_eca",
    "r18_cbam",
    "r18_procbam",
]:

    args = {
        **base_args,
        **get_backbone_variant(
            variant
        ),
    }

    learner = FeTrIL(args)

    learner._network.update_fc(
        14
    )

    learner._freeze_backbone()

    convnet_trainable = sum(
        p.requires_grad
        for p
        in learner
        ._network
        .convnet
        .parameters()
    )

    fc_trainable = sum(
        p.requires_grad
        for p
        in learner
        ._network
        .fc
        .parameters()
    )

    print(
        variant,
        "convnet:",
        convnet_trainable,
        "fc:",
        fc_trainable,
    )

    assert convnet_trainable == 0
    assert fc_trainable > 0


print(
    "FeTrIL frozen backbone: PASSED"
)

r18_eca convnet: 0 fc: 2
r18_cbam convnet: 0 fc: 2
r18_procbam convnet: 0 fc: 2
FeTrIL frozen backbone: PASSED


### Unit test SVM without training image

In [5]:
import numpy as np

from models.fetril import (
    FeatureDataset,
)


rng = np.random.RandomState(
    1993
)

num_classes = 14
feature_dim = 512


train_features = []
train_labels = []

test_features = []
test_labels = []


for c in range(
    num_classes
):

    center = np.zeros(
        feature_dim,
        dtype=np.float32,
    )

    center[c] = 10.0

    for _ in range(5):

        train_features.append(
            center
            + rng.normal(
                0,
                0.01,
                feature_dim,
            )
        )

        train_labels.append(c)

    for _ in range(2):

        test_features.append(
            center
            + rng.normal(
                0,
                0.01,
                feature_dim,
            )
        )

        test_labels.append(c)


train_set = FeatureDataset(
    np.asarray(
        train_features,
        dtype=np.float32,
    ),
    np.asarray(
        train_labels,
        dtype=np.int64,
    ),
)

test_set = FeatureDataset(
    np.asarray(
        test_features,
        dtype=np.float32,
    ),
    np.asarray(
        test_labels,
        dtype=np.int64,
    ),
)


args = {
    **base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}

learner = FeTrIL(args)

learner._feature_trainset = (
    train_set
)

learner._feature_testset = (
    test_set
)

learner._train_svm(
    train_set,
    test_set,
)

pred, true = learner._eval_svm()

top1 = (
    pred[:, 0] == true
).mean()


print(
    "Synthetic SVM top1:",
    top1,
)

assert top1 > 0.99

print(
    "FeTrIL SVM path: PASSED"
)

Synthetic SVM top1: 1.0
FeTrIL SVM path: PASSED


### Validasi reproducibility seed

In [6]:
import random
import numpy as np
import torch

from trainer import _set_random


_set_random(1993)

a_python = random.random()
a_numpy = np.random.rand()
a_torch = torch.rand(1)


_set_random(1993)

b_python = random.random()
b_numpy = np.random.rand()
b_torch = torch.rand(1)


assert a_python == b_python
assert a_numpy == b_numpy
assert torch.equal(
    a_torch,
    b_torch,
)


print(
    "Training seed reproducibility: PASSED"
)

Training seed reproducibility: PASSED


## Full Fetril smoke test

### Configure smoke test

In [7]:
from pathlib import Path

import gc
import time

import numpy as np
import torch

from torch.utils.data import (
    WeightedRandomSampler,
)

from models.fetril import FeTrIL
from trainer import _set_random
from utils.data_manager import DataManager

from thesis.backbones.variants import (
    get_backbone_variant,
)

In [8]:
SMOKE_SEED = 1993

smoke_args = {
    # =========================
    # Dataset
    # =========================
    "dataset": "aid",

    "dataset_root": str(
        PROJECT_ROOT
        / "datasets/raw/AID"
    ),

    "split_root": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993"
    ),

    "task_config_path": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993/T5_tasks.json"
    ),

    "shuffle": False,

    # =========================
    # Method
    # =========================
    "model_name": "fetril",

    "memory_size": 0,
    "fixed_memory": False,

    "fetril_classifier": "svm",

    "svm_c": 1.0,
    "svm_tol": 1e-4,

    # =========================
    # Smoke training
    # =========================
    "init_epochs": 1,

    "init_lr": 0.1,

    "init_weight_decay": 5e-4,

    # Tidak dipakai canonical SVM
    # pada incremental tasks,
    # tetapi tetap disediakan.
    "epochs": 1,

    "lr": 0.05,

    "weight_decay": 5e-4,

    # AID 600x600.
    # Ini SMOKE batch size,
    # belum final experimental batch.
    "batch_size": 4,

    # Notebook Windows:
    # 0 paling aman.
    "num_workers": 0,

    # =========================
    # Reproducibility
    # =========================
    "seed": SMOKE_SEED,

    # BaseLearner membutuhkan
    # list torch.device.
    "device": [
        torch.device(
            "cuda:0"
            if torch.cuda.is_available()
            else "cpu"
        )
    ],

    # =========================
    # Backbone
    # =========================
    **get_backbone_variant(
        "r18_none"
    ),
}

### Create Data Manager

In [9]:
_set_random(
    SMOKE_SEED
)


data_manager = DataManager(
    dataset_name=(
        smoke_args["dataset"]
    ),

    shuffle=(
        smoke_args["shuffle"]
    ),

    seed=(
        SMOKE_SEED
    ),

    init_cls=14,
    increment=4,

    dataset_root=(
        smoke_args[
            "dataset_root"
        ]
    ),

    split_root=(
        smoke_args[
            "split_root"
        ]
    ),

    task_config_path=(
        smoke_args[
            "task_config_path"
        ]
    ),
)


print(
    "Tasks:",
    data_manager.nb_tasks
)

print(
    "Task sizes:",
    [
        data_manager
        .get_task_size(i)

        for i in range(
            data_manager.nb_tasks
        )
    ]
)


assert (
    data_manager.nb_tasks
    == 5
)

assert [
    data_manager
    .get_task_size(i)

    for i in range(
        data_manager.nb_tasks
    )
] == [
    14,
    4,
    4,
    4,
    4,
]


print(
    "Smoke DataManager: PASSED"
)

Tasks: 5
Task sizes: [14, 4, 4, 4, 4]
Smoke DataManager: PASSED


### Initialize Fetril

In [10]:
learner = FeTrIL(
    smoke_args
)


print(
    "Initial known classes:",
    learner._known_classes
)

print(
    "Current task:",
    learner._cur_task
)

print(
    "Feature dim:",
    learner.feature_dim
)


assert (
    learner._known_classes
    == 0
)

assert (
    learner._cur_task
    == -1
)

assert (
    learner.feature_dim
    == 512
)


print(
    "FeTrIL initialization: PASSED"
)

Initial known classes: 0
Current task: -1
Feature dim: 512
FeTrIL initialization: PASSED


### Reset VRAM Statistics ?

In [11]:
if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


gc.collect()

36

### Run Task 0

In [12]:
task0_start = time.perf_counter()
learner.incremental_train(data_manager)
task0_seconds = (time.perf_counter() - task0_start)

print(
    "Task 0 runtime:",
    round(
        task0_seconds / 60,
        2,
    ),
    "minutes",
)

Task 0, Epoch 1/1 => Loss 2.677, Train_accy 15.04, Test_accy 22.10: 100%|██████████| 1/1 [01:45<00:00, 105.22s/it]


Task 0 runtime: 3.45 minutes


### Validate Task 0

In [13]:
print(
    "Current task:",
    learner._cur_task
)

print(
    "Total classes:",
    learner._total_classes
)

print(
    "Means:",
    len(
        learner._means
    )
)

print(
    "SVM classes:",
    learner
    ._svm_classifier
    .classes_
)

print(
    "Feature train shape:",
    learner
    ._feature_trainset
    .features
    .shape
)

print(
    "Feature test shape:",
    learner
    ._feature_testset
    .features
    .shape
)

print(
    "Train sampler:",
    type(
        learner
        .train_loader
        .sampler
    ).__name__,
)

print(
    "Exemplars:",
    learner.exemplar_size
)

Current task: 0
Total classes: 14
Means: 14
SVM classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
Feature train shape: torch.Size([3656, 512])
Feature test shape: torch.Size([914, 512])
Train sampler: WeightedRandomSampler
Exemplars: 0


In [14]:
assert learner._cur_task == 0

assert (
    learner._total_classes
    == 14
)

assert (
    len(
        learner._means
    )
    == 14
)

assert np.array_equal(
    learner
    ._svm_classifier
    .classes_,

    np.arange(14),
)

assert (
    learner
    ._feature_trainset
    .features
    .shape[1]
    == 512
)

assert (
    learner
    ._feature_testset
    .features
    .shape[1]
    == 512
)

assert isinstance(
    learner
    .train_loader
    .sampler,

    WeightedRandomSampler,
)

assert (
    learner.exemplar_size
    == 0
)

print(
    "FeTrIL Task 0 structure: PASSED"
)

FeTrIL Task 0 structure: PASSED


### Evaluate SVM Task 0

In [15]:
task0_accy, _ = (
    learner.eval_task()
)


print(
    "Task 0 SVM grouped:"
)

print(
    task0_accy[
        "grouped"
    ]
)

print(
    "Task 0 top-1:",
    task0_accy[
        "top1"
    ]
)

print(
    "Task 0 top-5:",
    task0_accy[
        "top5"
    ]
)


assert (
    0.0
    <= task0_accy[
        "top1"
    ]
    <= 100.0
)

assert np.isfinite(
    task0_accy[
        "top1"
    ]
)


print(
    "Task 0 SVM evaluation: PASSED"
)

Task 0 SVM grouped:
{'total': 47.7, '00-09': 47.28, '10-19': 48.62, 'old': 0, 'new': 47.7}
Task 0 top-1: 47.7
Task 0 top-5: 89.82
Task 0 SVM evaluation: PASSED


In [16]:
learner.after_task()


assert (
    learner._known_classes
    == 14
)

print(
    "Task 0 after_task: PASSED"
)

Task 0 after_task: PASSED


### Check peak of VRAM initial task

In [17]:
if torch.cuda.is_available():

    peak_allocated = (
        torch.cuda
        .max_memory_allocated()
        / 1024**3
    )

    peak_reserved = (
        torch.cuda
        .max_memory_reserved()
        / 1024**3
    )

    print(
        "Peak allocated:",
        round(
            peak_allocated,
            2,
        ),
        "GB",
    )

    print(
        "Peak reserved:",
        round(
            peak_reserved,
            2,
        ),
        "GB",
    )

Peak allocated: 0.75 GB
Peak reserved: 1.17 GB


### Run Task 1 seperately

In [18]:
task1_start = time.perf_counter()


learner.incremental_train(
    data_manager
)


task1_seconds = (
    time.perf_counter()
    - task1_start
)


print(
    "Task 1 runtime:",
    round(
        task1_seconds / 60,
        2,
    ),
    "minutes",
)

Task 1 runtime: 0.65 minutes


### Validate task 1

In [19]:
print(
    "Current task:",
    learner._cur_task
)

print(
    "Known before after_task:",
    learner._known_classes
)

print(
    "Total classes:",
    learner._total_classes
)

print(
    "Means:",
    len(
        learner._means
    )
)

print(
    "Relations:",
    learner._relations.shape
)

print(
    "Train loader:",
    learner.train_loader
)

print(
    "SVM classes:",
    learner
    ._svm_classifier
    .classes_
)

Current task: 1
Known before after_task: 14
Total classes: 18
Means: 18
Relations: (14,)
Train loader: None
SVM classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]


In [20]:
assert learner._cur_task == 1

assert (
    learner._known_classes
    == 14
)

assert (
    learner._total_classes
    == 18
)

assert (
    len(
        learner._means
    )
    == 18
)

assert (
    learner._relations.shape
    == (14,)
)

assert (
    learner.train_loader
    is None
)

assert np.array_equal(
    learner
    ._svm_classifier
    .classes_,

    np.arange(18),
)

In [21]:
trainable_backbone = sum(
    parameter.numel()

    for parameter
    in learner
    ._network
    .convnet
    .parameters()

    if parameter.requires_grad
)


print(
    "Trainable backbone params:",
    trainable_backbone
)


assert (
    trainable_backbone
    == 0
)

print(
    "FeTrIL Task 1 incremental "
    "behavior: PASSED"
)

Trainable backbone params: 0
FeTrIL Task 1 incremental behavior: PASSED


### Check pseudo feature set

In [22]:
feature_labels = (
    learner
    ._feature_trainset
    .labels
    .numpy()
)


unique_labels = np.unique(
    feature_labels
)


print(
    "Feature labels:",
    unique_labels
)

print(
    "Number of feature samples:",
    len(
        feature_labels
    )
)


assert np.array_equal(
    unique_labels,
    np.arange(18),
)


print(
    "Old + new feature classes: "
    "PASSED"
)

Feature labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]
Number of feature samples: 4176
Old + new feature classes: PASSED


### Evaluate task 1

In [23]:
task1_accy, _ = (
    learner.eval_task()
)


print(
    "Task 1 top-1:",
    task1_accy[
        "top1"
    ]
)

print(
    "Task 1 grouped:",
    task1_accy[
        "grouped"
    ]
)


assert np.isfinite(
    task1_accy[
        "top1"
    ]
)

learner.after_task()


assert (
    learner._known_classes
    == 18
)


print(
    "Task 1 evaluation: PASSED"
)

Task 1 top-1: 33.77
Task 1 grouped: {'total': 33.77, '00-09': 32.85, '10-19': 34.87, 'old': 32.93, 'new': 37.07}
Task 1 evaluation: PASSED


### Run the rest Task 2-4

In [24]:
top1_curve = [
    task0_accy["top1"],
    task1_accy["top1"],
]

task_runtime_minutes = [
    task0_seconds / 60,
    task1_seconds / 60,
]

In [25]:
for expected_task in range(
    2,
    data_manager.nb_tasks,
):

    start = time.perf_counter()

    learner.incremental_train(
        data_manager
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    accy, _ = (
        learner.eval_task()
    )

    expected_total = (
        data_manager
        .get_accumulate_tasksize(
            expected_task
        )
    )

    print()
    print(
        "=" * 60
    )

    print(
        "Task:",
        expected_task
    )

    print(
        "Total classes:",
        learner._total_classes
    )

    print(
        "Means:",
        len(
            learner._means
        )
    )

    print(
        "SVM classes:",
        len(
            learner
            ._svm_classifier
            .classes_
        )
    )

    print(
        "Top-1:",
        accy["top1"]
    )

    print(
        "Runtime:",
        round(
            elapsed / 60,
            2,
        ),
        "minutes",
    )


    assert (
        learner._cur_task
        == expected_task
    )

    assert (
        learner._total_classes
        == expected_total
    )

    assert (
        len(
            learner._means
        )
        == expected_total
    )

    assert (
        len(
            learner
            ._svm_classifier
            .classes_
        )
        == expected_total
    )

    assert (
        learner.train_loader
        is None
    )

    assert (
        learner.exemplar_size
        == 0
    )

    assert np.isfinite(
        accy["top1"]
    )


    top1_curve.append(
        accy["top1"]
    )

    task_runtime_minutes.append(
        elapsed / 60
    )


    learner.after_task()


Task: 2
Total classes: 22
Means: 22
SVM classes: 22
Top-1: 28.76
Runtime: 0.79 minutes

Task: 3
Total classes: 26
Means: 26
SVM classes: 26
Top-1: 26.98
Runtime: 0.88 minutes

Task: 4
Total classes: 30
Means: 30
SVM classes: 30
Top-1: 26.1
Runtime: 1.01 minutes


### Final smoke validation

In [26]:
print(
    "Top-1 curve:",
    top1_curve
)

print(
    "Task runtime minutes:",
    [
        round(x, 2)
        for x
        in task_runtime_minutes
    ]
)


average_incremental_accuracy = (
    float(
        np.mean(
            top1_curve
        )
    )
)

final_accuracy = (
    float(
        top1_curve[-1]
    )
)


print(
    "Smoke AIA:",
    round(
        average_incremental_accuracy,
        2,
    )
)

print(
    "Smoke final accuracy:",
    round(
        final_accuracy,
        2,
    )
)


assert (
    len(
        top1_curve
    )
    == 5
)

assert (
    learner._known_classes
    == 30
)

assert (
    learner._total_classes
    == 30
)

assert (
    len(
        learner._means
    )
    == 30
)

assert (
    len(
        learner._svm_accs
    )
    == 5
)

assert (
    learner.exemplar_size
    == 0
)


print(
    "FULL FeTrIL T5 smoke test: "
    "PASSED"
)

Top-1 curve: [47.7, 33.77, 28.76, 26.98, 26.1]
Task runtime minutes: [3.45, 0.65, 0.79, 0.88, 1.01]
Smoke AIA: 32.66
Smoke final accuracy: 26.1
FULL FeTrIL T5 smoke test: PASSED


## Benchmark with 8 backbone and AID dataset

### Setup

In [27]:
import gc
import time

import numpy as np
import pandas as pd
import torch

from torch import optim
from torch.nn import functional as F
from torch.utils.data import (
    WeightedRandomSampler,
)

from models.fetril import FeTrIL

from thesis.backbones.variants import (
    get_all_backbone_variants,
)

from thesis.data.loaders import (
    build_train_loader,
)

from trainer import _set_random

### Take AID Task 0

In [28]:
task0_classes = (
    data_manager.get_task_size(0)
)

task0_train_dataset = (
    data_manager.get_dataset(
        np.arange(
            0,
            task0_classes,
        ),
        source="train",
        mode="train",
    )
)


print(
    "Task 0 classes:",
    task0_classes
)

print(
    "Task 0 samples:",
    len(
        task0_train_dataset
    )
)


assert task0_classes == 14

assert (
    len(
        task0_train_dataset
    )
    == 3656
)


print(
    "Benchmark dataset: PASSED"
)

Task 0 classes: 14
Task 0 samples: 3656
Benchmark dataset: PASSED


### Benchmark function



In [32]:
BENCH_BATCH_SIZE = 4
WARMUP_STEPS = 3
BENCH_STEPS = 30


def benchmark_fetril_variant(
    variant_name,
    backbone_config,
):

    _set_random(
        SMOKE_SEED
    )

    args = {
        **smoke_args,
        **backbone_config,

        "batch_size":
            BENCH_BATCH_SIZE,

        "num_workers":
            0,
    }


    learner = FeTrIL(
        args
    )

    network = (
        learner._network
    )

    network.update_fc(
        nb_classes=14
    )

    device = args[
        "device"
    ][0]

    network = network.to(
        device
    )

    network.train()


    loader = build_train_loader(
        dataset=(
            task0_train_dataset
        ),

        dataset_name="aid",

        batch_size=(
            BENCH_BATCH_SIZE
        ),

        num_workers=0,

        seed=(
            SMOKE_SEED
        ),
    )


    assert isinstance(
        loader.sampler,
        WeightedRandomSampler,
    )


    optimizer = optim.SGD(
        network.parameters(),

        lr=0.1,

        momentum=0.9,

        weight_decay=5e-4,
    )


    iterator = iter(loader)


    # =========================
    # Warm-up
    # =========================
    for _ in range(
        WARMUP_STEPS
    ):

        _, inputs, targets = next(
            iterator
        )

        targets = targets.long()

        inputs = inputs.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = network(
            inputs
        )["logits"]

        loss = F.cross_entropy(
            logits,
            targets,
        )

        loss.backward()

        optimizer.step()


    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()


    # =========================
    # Timed section
    # =========================
    start = time.perf_counter()

    last_loss = None


    for _ in range(
        BENCH_STEPS
    ):

        _, inputs, targets = next(
            iterator
        )

        targets = targets.long()

        inputs = inputs.to(
            device,
            non_blocking=True,
        )

        targets = targets.to(
            device,
            non_blocking=True,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = network(
            inputs
        )["logits"]


        loss = F.cross_entropy(
            logits,
            targets,
        )


        loss.backward()

        optimizer.step()

        last_loss = float(
            loss.detach().cpu()
        )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    seconds_per_step = (
        elapsed
        / BENCH_STEPS
    )


    projected_epoch_seconds = (
        seconds_per_step
        * len(loader)
    )


    attention_grad_params = sum(
        1
        for name, parameter
        in network.named_parameters()

        if (
            ".attention." in name
            and
            parameter.grad is not None
        )
    )


    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )

    else:

        peak_allocated = 0.0
        peak_reserved = 0.0


    result = {
        "variant":
            variant_name,

        "batch_size":
            BENCH_BATCH_SIZE,

        "steps_per_epoch":
            len(loader),

        "sec_per_step":
            seconds_per_step,

        "projected_epoch_min":
            projected_epoch_seconds
            / 60,

        "peak_allocated_gb":
            peak_allocated,

        "peak_reserved_gb":
            peak_reserved,

        "attention_grad_params":
            attention_grad_params,

        "last_loss":
            last_loss,
    }


    assert np.isfinite(
        last_loss
    )


    attention_type = (
        backbone_config[
            "attention_type"
        ]
    )


    if attention_type == "none":

        assert (
            attention_grad_params
            == 0
        )

    else:

        assert (
            attention_grad_params
            > 0
        )


    del learner
    del network
    del optimizer
    del loader
    del iterator
    del inputs
    del targets
    del logits
    del loss


    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return result

### Run all 8 variants

In [33]:
benchmark_results = []


variants = (
    get_all_backbone_variants()
)


for variant_name, config in (
    variants.items()
):

    print()
    print(
        "=" * 60
    )

    print(
        "Benchmark:",
        variant_name
    )


    result = (
        benchmark_fetril_variant(
            variant_name,
            config,
        )
    )


    benchmark_results.append(
        result
    )


    print(
        "sec/step:",
        round(
            result[
                "sec_per_step"
            ],
            4,
        )
    )

    print(
        "projected epoch:",
        round(
            result[
                "projected_epoch_min"
            ],
            2,
        ),
        "min",
    )

    print(
        "peak allocated:",
        round(
            result[
                "peak_allocated_gb"
            ],
            2,
        ),
        "GB",
    )

    print(
        "attention gradients:",
        result[
            "attention_grad_params"
        ],
    )


Benchmark: r18_none
sec/step: 0.098
projected epoch: 1.49 min
peak allocated: 1.46 GB
attention gradients: 0

Benchmark: r18_eca
sec/step: 0.096
projected epoch: 1.46 min
peak allocated: 1.53 GB
attention gradients: 8

Benchmark: r18_cbam
sec/step: 0.1139
projected epoch: 1.74 min
peak allocated: 1.62 GB
attention gradients: 56

Benchmark: r18_procbam
sec/step: 0.1197
projected epoch: 1.82 min
peak allocated: 1.62 GB
attention gradients: 88

Benchmark: r34_none
sec/step: 0.1358
projected epoch: 2.07 min
peak allocated: 1.81 GB
attention gradients: 0

Benchmark: r34_eca
sec/step: 0.1463
projected epoch: 2.23 min
peak allocated: 1.95 GB
attention gradients: 16

Benchmark: r34_cbam
sec/step: 0.1775
projected epoch: 2.7 min
peak allocated: 2.11 GB
attention gradients: 112

Benchmark: r34_procbam
sec/step: 0.1875
projected epoch: 2.86 min
peak allocated: 2.11 GB
attention gradients: 176


### Show runtime table

In [34]:
benchmark_df = pd.DataFrame(
    benchmark_results
)


display(
    benchmark_df[
        [
            "variant",
            "sec_per_step",
            "projected_epoch_min",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "attention_grad_params",
        ]
    ]
)

,variant,sec_per_step,projected_epoch_min,peak_allocated_gb,peak_reserved_gb,attention_grad_params
0,r18_none,0.098037,1.493425,1.455810,1.804688,0
1,r18_eca,0.096013,1.462599,1.531880,1.974609,8
2,r18_cbam,0.113928,1.735510,1.618556,2.080078,56
3,r18_procbam,0.119666,1.822906,1.616637,2.062500,88
4,r34_none,0.135784,2.068438,1.811358,2.271484,0
5,r34_eca,0.146279,2.228309,1.951416,2.421875,16
6,r34_cbam,0.177545,2.704598,2.106895,2.576172,112
7,r34_procbam,0.187524,2.856620,2.110024,2.574219,176


### Calculate slowdown on R18 None

In [35]:
baseline_step_time = float(
    benchmark_df.loc[
        benchmark_df[
            "variant"
        ]
        == "r18_none",

        "sec_per_step",
    ].iloc[0]
)


benchmark_df[
    "runtime_multiplier"
] = (
    benchmark_df[
        "sec_per_step"
    ]
    / baseline_step_time
)


display(
    benchmark_df[
        [
            "variant",
            "sec_per_step",
            "runtime_multiplier",
            "peak_allocated_gb",
        ]
    ]
)

,variant,sec_per_step,runtime_multiplier,peak_allocated_gb
0,r18_none,0.098037,1.000000,1.455810
1,r18_eca,0.096013,0.979358,1.531880
2,r18_cbam,0.113928,1.162100,1.618556
3,r18_procbam,0.119666,1.220621,1.616637
4,r34_none,0.135784,1.385029,1.811358
5,r34_eca,0.146279,1.492079,1.951416
6,r34_cbam,0.177545,1.811003,2.106895
7,r34_procbam,0.187524,1.912797,2.110024


### Estimate FeTriL runtime based on provided benchmark

In [36]:
BASELINE_T5_MIN_HOURS = 5.0
BASELINE_T5_MAX_HOURS = 5.9

BASELINE_T10_MIN_HOURS = 4.3
BASELINE_T10_MAX_HOURS = 5.1

In [37]:
benchmark_df[
    "T5_est_min_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T5_MIN_HOURS
)


benchmark_df[
    "T5_est_max_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T5_MAX_HOURS
)


benchmark_df[
    "T10_est_min_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T10_MIN_HOURS
)


benchmark_df[
    "T10_est_max_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T10_MAX_HOURS
)


display(
    benchmark_df[
        [
            "variant",
            "runtime_multiplier",
            "T5_est_min_h",
            "T5_est_max_h",
            "T10_est_min_h",
            "T10_est_max_h",
            "peak_allocated_gb",
        ]
    ]
)

,variant,runtime_multiplier,T5_est_min_h,T5_est_max_h,T10_est_min_h,T10_est_max_h,peak_allocated_gb
0,r18_none,1.000000,5.000000,5.900000,4.300000,5.100000,1.455810
1,r18_eca,0.979358,4.896792,5.778214,4.211241,4.994727,1.531880
2,r18_cbam,1.162100,5.810500,6.856390,4.997030,5.926710,1.618556
3,r18_procbam,1.220621,6.103105,7.201663,5.248670,6.225167,1.616637
4,r34_none,1.385029,6.925146,8.171672,5.955626,7.063649,1.811358
5,r34_eca,1.492079,7.460397,8.803268,6.415941,7.609604,1.951416
6,r34_cbam,1.811003,9.055015,10.684918,7.787313,9.236116,2.106895
7,r34_procbam,1.912797,9.563985,11.285502,8.225027,9.755265,2.110024


In [38]:
total_min_hours = (
    benchmark_df[
        "T5_est_min_h"
    ].sum()
    +
    benchmark_df[
        "T10_est_min_h"
    ].sum()
)


total_max_hours = (
    benchmark_df[
        "T5_est_max_h"
    ].sum()
    +
    benchmark_df[
        "T10_est_max_h"
    ].sum()
)


print(
    "Estimated FeTrIL AID GPU-hours:"
)

print(
    round(
        total_min_hours,
        1,
    ),
    "-",
    round(
        total_max_hours,
        1,
    ),
    "hours",
)

Estimated FeTrIL AID GPU-hours:
102.0 - 120.6 hours
